In [0]:
from pyspark.sql import functions as F

current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
dbutils.widgets.text("target_batch", "batch_1", "Target Landing Batch Directory")

base_path = dbutils.widgets.get("base_path")
target_batch = dbutils.widgets.get("target_batch")

source_landing_path = f"{base_path}/landing/{target_batch}"
bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
schema_location = f"{base_path}/bronze/_schema"
checkpoint_location = f"{base_path}/bronze/_checkpoint"

# Ingest landing JSON records incrementally via Auto Loader
# - cloudFiles.inferColumnTypes: infers primitive types automatically across incoming JSON fields
# - cloudFiles.rescuedDataColumn: captures unexpected extra fields or data type mismatches without breaking streaming pipelines
# - _metadata.file_path: retrieves input file path natively under Unity Catalog governance
df_bronze_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_landing_path)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# Execute trigger-once append stream to Bronze Delta storage
query = (
    df_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .start(bronze_table_path)
)

query.awaitTermination()

In [0]:
current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")

bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
df_bronze = spark.read.format("delta").load(bronze_table_path)

print("=== BRONZE TABLE METADATA & INGESTION VERIFICATION ===")
print(f"Total Records Ingested: {df_bronze.count()}")
print("Schema Columns:", df_bronze.columns)

df_bronze.select(
    "event_id", "magnitude", "place", "_ingested_at", "_source_file", "_rescued_data"
).show(5, truncate=False)